# Orchestrator-workers workflow with Pydantic AI

The orchestrator-worker pattern works well for tasks where you don't know the required subtasks beforehand. An orchestrator determines the subtasks, workers execute them in parallel, and a synthesizer combines the results.

```mermaid
flowchart LR
    In([In]) --> Orch[Orchestrator]

    Orch -.-> LLM1["LLM Call 1"]
    Orch -.-> LLM2["LLM Call 2"]
    Orch -.-> LLM3["LLM Call 3"]

    LLM1 -.-> Synth[Synthesizer]
    LLM2 -.-> Synth
    LLM3 -.-> Synth

    Synth --> Out([Out])
```

The key difference from parallelization is that the subtasks are **dynamically determined** by the orchestrator, not predefined.

**Examples:**
- Coding tools making changes to multiple files at once
- Searching multiple sources and synthesizing the results

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
import asyncio

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent

load_dotenv()

## Vanilla workflow

The orchestrator generates article sections, workers write each section in parallel, and a synthesizer combines them.

In [ ]:
class Section(BaseModel):
    name: str = Field(description="The name of the section")
    description: str = Field(description="The description of the section")


class CompletedSection(BaseModel):
    name: str = Field(description="The name of the section")
    content: str = Field(description="The content of the section")


class Sections(BaseModel):
    sections: list[Section] = Field(description="The sections of the article")


orchestrator = Agent(
    "openai:gpt-5-mini",
    output_type=Sections,
    system_prompt=(
        "You are an expert writer specialized in SEO. Provided with a topic, "
        "you will generate the sections for a short article."
    ),
)

worker = Agent(
    "openai:gpt-5-mini",
    output_type=CompletedSection,
    system_prompt=(
        "You are an expert writer specialized in SEO. Provided with a topic and a section, "
        "you will generate the content of the section."
    ),
)


def synthesizer(sections: list[CompletedSection]) -> str:
    return "\n\n".join([section.content for section in sections])


async def run_workflow(topic: str) -> str:
    # 1. Orchestrator determines sections
    plan = await orchestrator.run(
        f"Generate the sections for a short article about {topic}"
    )

    # 2. Workers write each section in parallel
    tasks = [
        worker.run(
            f"Write the section {section.name} about {topic} "
            f"with the following description: {section.description}"
        )
        for section in plan.output.sections
    ]
    completed = await asyncio.gather(*tasks)

    # 3. Synthesizer combines results
    return synthesizer([c.output for c in completed])


output = await run_workflow("Artificial Intelligence")
print(output)

## Exercise

Build an orchestrator-worker workflow that generates a travel guide. The orchestrator should determine which sections to cover (e.g., transportation, food, attractions), and each worker writes its section.

In [ ]:
travel_orchestrator = Agent(
    "openai:gpt-5-mini",
    output_type=Sections,
    system_prompt=(
        "You are a travel editor. Given a destination, plan the sections "
        "for a concise travel guide."
    ),
)

travel_worker = Agent(
    "openai:gpt-5-mini",
    output_type=CompletedSection,
    system_prompt=(
        "You are a travel writer. Given a destination and section brief, "
        "write that section of the travel guide."
    ),
)


def synthesize_travel_guide(sections: list[CompletedSection]) -> str:
    return "\n\n".join(f"## {section.name}\n{section.content}" for section in sections)


async def run_travel_guide(destination: str) -> str:
    plan = await travel_orchestrator.run(
        "Create 4 sections for a travel guide about "
        f"{destination}. Include transportation, food, attractions, "
        "and practical tips when relevant."
    )

    tasks = [
        travel_worker.run(
            f"Destination: {destination}\n"
            f"Section name: {section.name}\n"
            f"Description: {section.description}"
        )
        for section in plan.output.sections
    ]
    completed = await asyncio.gather(*tasks)
    return synthesize_travel_guide([c.output for c in completed])

In [ ]:
travel_guide = await run_travel_guide("Tokyo")
print(travel_guide)